In [6]:
#과제3번

import csv

class Heap:
    def __init__(self, *args):
        if len(args) != 0:
            self.__A = args[0]
        else:
            self.__A = []

    def insert(self, x):
        self.__A.append(x)
        self.__percolateUp(len(self.__A) - 1)

    def __percolateUp(self, i: int):
        parent = (i - 1) // 2
        if i > 0 and self.__A[i] > self.__A[parent]:
            self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
            self.__percolateUp(parent)

    def deleteMax(self):
        if not self.isEmpty():
            max = self.__A[0]
            self.__A[0] = self.__A.pop()
            self.__percolateDown(0)
            return max
        else:
            return None

    def __percolateDown(self, i: int):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self.__percolateDown(child)

    def max(self):
        return self.__A[0]

    def buildHeap(self):
        for i in range((len(self.__A) - 2) // 2, -1, -1):
            self.__percolateDown(i)

    def isEmpty(self) -> bool:
        return len(self.__A) == 0

    def clear(self):
        self.__A = []

    def size(self) -> int:
        return len(self.__A)


def load_birthdays(filepath: str) -> list[tuple]:
    friends = []
    with open(filepath, encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        next(reader)  # 헤더 스킵
        for row in reader:
            if len(row) < 4:
                continue
            name, year, month, day = row[0], row[1], row[2], row[3]
            if name and year and month and day:
                friends.append((name, int(year), int(month), int(day)))
    return friends

def top10_latest_birthdays(filepath: str):
    friends = load_birthdays(filepath)

    heap = Heap()
    for name, year, month, day in friends:
        date_key = month * 100 + day  # 생일(월일)만 기준
        heap.insert((date_key, name))

    results = []
    for _ in range(min(10, heap.size())):
        date_key, name = heap.deleteMax()
        month = date_key // 100
        day = date_key % 100
        results.append((name, month, day))

    print("=== 생일이 늦은 순 TOP 10 ===")
    for rank, (name, month, day) in enumerate(results, 1):
        print(f"{rank:2}. {name}  {month}월 {day}일")

# 실행
top10_latest_birthdays("birthday.csv")

=== 생일이 늦은 순 TOP 10 ===
 1. 이윤서  12월 27일
 2. 정희원  12월 21일
 3. 김효린  12월 16일
 4. 이예은  12월 9일
 5. 엥흐툽신 엥흐톨  12월 5일
 6. 정인선  11월 24일
 7. 김주영  11월 20일
 8. 진가연  11월 12일
 9. 전은빈  11월 7일
10. 황다원  10월 15일


In [7]:
#과제4번

import csv

# ── BidirectNode ──────────────────────────────────────────
class BidirectNode:
    def __init__(self, item, prev, next):
        self.item = item
        self.prev = prev
        self.next = next

# ── CircularDoublyLinkedList ───────────────────────────────
class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0

    def append(self, newItem) -> None:
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def getNode(self, i: int) -> BidirectNode:
        curr = self.__head
        for _ in range(i + 1):
            curr = curr.next
        return curr

    def isEmpty(self) -> bool:
        return self.__numItems == 0

    def size(self) -> int:
        return self.__numItems

    def __iter__(self):
        return CircularDoublyLinkedListIterator(self)

class CircularDoublyLinkedListIterator:
    def __init__(self, alist):
        self.__head = alist.getNode(-1)
        self.iterPosition = self.__head.next

    def __next__(self):
        if self.iterPosition == self.__head:
            raise StopIteration
        item = self.iterPosition.item
        self.iterPosition = self.iterPosition.next
        return item

# ── CSV 읽기 및 출력 ───────────────────────────────────────
def load_from_csv(filepath: str, start_row: int) -> CircularDoublyLinkedList:
    """
    start_row: 실제 파일 줄 번호 (1-indexed, 헤더 포함)
    예) 33이면 파일의 33번째 줄부터 읽음
    """
    cdll = CircularDoublyLinkedList()

    with open(filepath, encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        for _ in range(start_row - 1):  # start_row 이전 줄 스킵
            next(reader, None)

        for row in reader:
            if len(row) < 4:
                continue

            name = row[0].strip()
            year  = row[1].strip()
            month = row[2].strip()
            day   = row[3].strip()

            # 생년월일 예외처리: 하나라도 비어있으면 스킵
            if not year or not month or not day:
                print(f"[예외] {name} - 생년월일 정보 없음, 건너뜀")
                continue

            try:
                birthday = (name, int(year), int(month), int(day))
                cdll.append(birthday)
            except ValueError:
                print(f"[예외] {name} - 생년월일 형식 오류 ({year}, {month}, {day}), 건너뜀")

    return cdll

def print_list(cdll: CircularDoublyLinkedList) -> None:
    print("=== 4조 조원 이름 및 생년월일 ===")
    for name, year, month, day in cdll:
        print(f"{name}  {year}년 {month}월 {day}일")

# 실행
cdll = load_from_csv("birthday.csv", start_row=33)  # 33번째 줄부터 읽기
print_list(cdll)

=== 4조 조원 이름 및 생년월일 ===
황다원  2004년 10월 15일
박다인  2005년 1월 19일
안예원  2004년 7월 9일
정서하  2006년 7월 20일
김우현  2006년 9월 1일
양시언  2006년 4월 18일
유가현  2006년 8월 22일
이하연  2006년 9월 22일
장예은  2006년 3월 17일
전은빈  2006년 11월 7일
정세희  2004년 1월 27일
이예은  2006년 12월 9일


과제 5번 풀이

1. 가질 수 있다.
2. 마지막 원소가 항상 가장 작은 값을 갖는 건 아니다.
3. 전체 원소 개수의 절반 n/2
4. 최악의 경우: Θ(log n), 최선의 경우: Θ(1)
5. 간단하다. 배열의 마지막 요소만 삭제하면 된다.
6. 더 비효율적이다. 기존 방식의 점근적 시간복잡도는 O(n)인 것에 비해 이 방식은 점근적 시간복잡도가 O(n logn)이 된다.
7. 부모 노드보다 작거나 같아질 때까지 비교해서 스며오르기를 반복하면 된다.